# Ejercicio 6 — Benchmark Parquet directo vs tabla DuckDB

El benchmark lo ejecuta `scripts/benchmark.py` (consultas en `sql/04_benchmark.sql`). Este notebook lo lanza (opcional) y presenta los resultados guardados en `data/processed/benchmark_resultados.csv`.

In [ ]:
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

EJECUTAR = False   # cambiar a True para (re)ejecutar el benchmark (tarda: construye la tabla y repite consultas)
if EJECUTAR:
    subprocess.run([sys.executable, "../scripts/benchmark.py", "--rebuild", "--reps", "5"], check=True)

In [ ]:
df = pd.read_csv("../data/processed/benchmark_resultados.csv")
df.head()

## 6.7 Tabla de resultados (mediana de las repeticiones, en segundos)

In [ ]:
med = (df.groupby(["periodos", "filas", "consulta", "estrategia"])["segundos"].median()
         .unstack("estrategia").reset_index())
med["parquet/tabla"] = (med["parquet"] / med["tabla"]).round(2)
med.round(4)

## Tiempo vs. cantidad de datos (6.6)

In [ ]:
consultas = sorted(med.consulta.unique())
fig, axs = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
for ax, c in zip(axs.flatten(), consultas):
    g = med[med.consulta == c].sort_values("filas")
    ax.plot(g.filas / 1e6, g.parquet, marker="o", label="Parquet directo")
    ax.plot(g.filas / 1e6, g.tabla, marker="s", label="Tabla DuckDB")
    ax.set_title(c, fontsize=9); ax.set_xlabel("millones de filas"); ax.set_ylabel("segundos"); ax.grid(alpha=.3)
axs.flatten()[0].legend()
for ax in axs.flatten()[len(consultas):]: ax.axis("off")
plt.tight_layout(); plt.show()

## 6.9 Análisis

_(completar con sus resultados: ¿en qué consultas gana la tabla y por cuánto? ¿cómo cambia con el volumen? ¿qué costo tiene construir la tabla? Ver guía en `docs/06_benchmark.md`.)_